In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import pymn
import anndata as ad
import scipy.sparse as sp

import os
import gc
import re
import time
import datetime
import resource
from pathlib import Path

In [2]:
start_time = time.time()

# -------------------------
# Core paths
# -------------------------

A1_h5ad = "/sbgenomics/project-files/GEN_A1/GEN_A1_pass2.h5ad"
A2_h5ad = "/sbgenomics/project-files/GEN_A2/250917_GEN_A2/GEN_A2_pass3.h5ad"
A3_h5ad = "/sbgenomics/project-files/GEN_A3/GEN_A3_pass2.h5ad"  # A3 needs cortex filtering

annotation_mapping_path = (
    "/sbgenomics/project-files//mappings_for_combined_five_PFC.20260130_210942/"
    "subtype_mappings.csv"
)

# Previous pyMN run folders used to discover GEN_A4/5/13/16 etc.
BASE_DIR = Path("/sbgenomics/project-files/python_Metaneighbor_runs/")
OUTPUT_DIR = Path(str(BASE_DIR).replace("project-files", "output-files", 1))
SEARCH_DIRS = [d for d in [BASE_DIR, OUTPUT_DIR] if d.exists()]

# -------------------------
# Sampling / filtering settings
# -------------------------

# Matches the old target-loading script.
# Set to None if memory allows loading all cells.
MAX_CELLS_PER_DISCOVERED_TARGET = 10_000
#MAX_CELLS_PER_DISCOVERED_TARGET = 20_000

GEN_A1_SAMPLE_FRACTION = MAX_CELLS_PER_DISCOVERED_TARGET/8_000_000
GEN_A2_SAMPLE_FRACTION = MAX_CELLS_PER_DISCOVERED_TARGET/1_500_000
GEN_A3_SAMPLE_FRACTION = 1


RANDOM_SEED = 42

# These are loaded directly below, so do not also load them from discovered old runs.
STUDY_IDS_ALREADY_LOADED = {"GEN_A1", "GEN_A2", "GEN_A3", "GEN_A3_ctx"}

# If an old GEN_A2 run appears in the discovered targets, exclude it too.
# This replaces the old "internal_dataset" logic.
EXCLUDE_DISCOVERED_STUDY_IDS = STUDY_IDS_ALREADY_LOADED | {"GEN_A2"}

result_folder = (
    "/sbgenomics/output-files/"
    + "GEN_A_all.cortex.subclass."
    + str(round(time.time()))
)

os.mkdir(result_folder)
print(result_folder)

/sbgenomics/output-files/GEN_A_all.cortex.subclass.1778701183


In [3]:
annotation_mapping_file = pd.read_csv(annotation_mapping_path)

annotation_mapping_file = annotation_mapping_file[
    ["subtype", "subclass_annotation_v1", "dataset"]
].copy()

annotation_mapping_file.head()

,subtype,subclass_annotation_v1,dataset
0,12_4_1,Adaptive_Immune,GEN_A1
1,12_1_1,Adaptive_Immune,GEN_A1
2,12_1_2,Adaptive_Immune,GEN_A1
3,12_2_1,Adaptive_Immune,GEN_A1
4,12_1_3,Adaptive_Immune,GEN_A1


In [4]:
set(annotation_mapping_file.subclass_annotation_v1.to_list())

{'Adaptive_Immune',
 'Astro',
 'EN_L2-3_IT',
 'EN_L3-5_IT_1',
 'EN_L3-5_IT_2',
 'EN_L3-5_IT_3',
 'EN_L5-6_NP',
 'EN_L5_ET',
 'EN_L6B',
 'EN_L6_CT',
 'EN_L6_IT_1',
 'EN_L6_IT_2',
 'Endo',
 'IN_CGE_KCNG1',
 'IN_CGE_LAMP5_RELN',
 'IN_CGE_VIP',
 'IN_LAMP5_LHX6',
 'IN_MGE_PVALB',
 'IN_MGE_PVALB_CHC',
 'IN_MGE_SST',
 'Micro-PVM',
 'OPC',
 'Oligo',
 'PC',
 'SMC',
 'VLMC'}

In [5]:
def load_h5ad_sample(
    h5ad_path,
    study_id,
    sample_fraction=1.0,
    random_seed=42,
    use_raw=True,
    set_gene_id_index=True,
):
    """
    Load h5ad file with optional random sampling of cells.

    Uses raw.X/raw.var when available and use_raw=True.
    """
    backed = sc.read_h5ad(h5ad_path, backed="r")
    n_cells = backed.n_obs

    if sample_fraction < 1.0:
        rng = np.random.default_rng(random_seed)
        sample_size = int(n_cells * sample_fraction)
        cell_indices = np.sort(
            rng.choice(n_cells, size=sample_size, replace=False)
        )
    else:
        cell_indices = np.arange(n_cells)

    if use_raw and backed.raw is not None:
        X = backed.raw.X[cell_indices, :]
        obs = backed.obs.iloc[cell_indices].copy()
        var = backed.var.copy()
        adata = sc.AnnData(X=X, obs=obs, var=var)
    else:
        adata = backed[cell_indices, :].to_memory()

    backed.file.close()

    adata = adata.to_memory()

    if set_gene_id_index and "gene_id" in adata.var.columns:
        adata.var = adata.var.set_index("gene_id", drop=False)

    adata.obs["study_id"] = study_id

    return adata

In [10]:
def load_A3_cortex(
    h5ad_path,
    study_id="GEN_A3_ctx",
    sample_fraction=1.0,
    random_seed=42,
    cortex_regions=("PFC"),#, "PMC", "PVC"),
    set_gene_id_index=True,
):
    """
    Load A3 cortex cells only.

    This follows the original A3 workflow using X rather than raw.X.
    """
    backed = sc.read_h5ad(h5ad_path, backed="r")

    # Make robust to accidentally passing a single string
    if isinstance(cortex_regions, str):
        cortex_regions = [cortex_regions]
    else:
        cortex_regions = list(cortex_regions)
        
    region_mask = backed.obs["brain_region"].isin(cortex_regions).to_numpy()
    region_indices = np.where(region_mask)[0]

    print(
        f"Found {len(region_indices):,} cells with {cortex_regions} "
        f"in brain_region out of {backed.n_obs:,} total"
    )

    if sample_fraction < 1.0:
        rng = np.random.default_rng(random_seed)
        sample_size = int(len(region_indices) * sample_fraction)
        region_indices = np.sort(
            rng.choice(region_indices, size=sample_size, replace=False)
        )

    X_subset = backed.X[region_indices, :]
    obs_subset = backed.obs.iloc[region_indices].copy()
    var_subset = backed.var.copy()

    adata = sc.AnnData(
        X=X_subset,
        obs=obs_subset,
        var=var_subset,
    ).to_memory()

    backed.file.close()

    if set_gene_id_index and "gene_id" in adata.var.columns:
        adata.var = adata.var.set_index("gene_id", drop=False)

    adata.obs["study_id"] = study_id

    print(f"Loaded {adata.n_obs:,} A3 cortex cells into memory")

    return adata

In [7]:
def add_subclass_annotation_inplace(
    adata,
    mapping: pd.DataFrame,
    *,
    subtype_col: str = "subtype",
    subclass_col: str = "subclass_annotation_v1",
    dataset_col: str = "dataset",
    dataset_key=None,
    out_col: str = "subclass_annotation_v1",
    drop_unmapped: bool = True,
):
    """
    Map adata.obs[subtype_col] to subclass_annotation_v1 using the A1-A3 mapping file.
    """
    cols = [subtype_col, subclass_col]

    if dataset_col in mapping.columns:
        cols.append(dataset_col)

    m = mapping.loc[:, cols].drop_duplicates()

    if dataset_key is not None:
        if dataset_col not in m.columns:
            raise ValueError(
                f"`dataset_key` was provided, but mapping has no {dataset_col!r} column."
            )
        m = m[m[dataset_col].astype(str).eq(str(dataset_key))]

    mapper = (
        m.dropna(subset=[subtype_col, subclass_col])
         .assign(**{subtype_col: lambda x: x[subtype_col].astype(str)})
         .set_index(subtype_col)[subclass_col]
    )

    adata.obs[out_col] = (
        adata.obs[subtype_col]
        .astype(str)
        .map(mapper)
    )

    n_total = adata.n_obs
    n_unmapped = int(adata.obs[out_col].isna().sum())
    n_mapped = n_total - n_unmapped

    print(
        f"[{adata.obs['study_id'].iloc[0]}] mapped {n_mapped:,} / {n_total:,} cells; "
        f"unmapped {n_unmapped:,} ({n_unmapped / max(1, n_total):.1%})"
    )

    if drop_unmapped and n_unmapped > 0:
        mask = adata.obs[out_col].notna().to_numpy()
        adata._inplace_subset_obs(mask)

    return adata

In [9]:
adatas = {}

A1 = load_h5ad_sample(
    A1_h5ad,
    "GEN_A1",
    sample_fraction=GEN_A1_SAMPLE_FRACTION,
    random_seed=RANDOM_SEED,
)

A2 = load_h5ad_sample(
    A2_h5ad,
    "GEN_A2",
    sample_fraction=GEN_A2_SAMPLE_FRACTION,
    random_seed=RANDOM_SEED,
)

TypeError: only list-like objects are allowed to be passed to isin(), you passed a `str`

In [11]:
A3 = load_A3_cortex(
    A3_h5ad,
    study_id="GEN_A3_ctx",
    sample_fraction=GEN_A3_SAMPLE_FRACTION,
    random_seed=RANDOM_SEED,
)

Found 530,170 cells with ['PFC'] in brain_region out of 2,327,968 total
Loaded 530,170 A3 cortex cells into memory


In [12]:
add_subclass_annotation_inplace(
    A1,
    annotation_mapping_file,
    dataset_key="GEN_A1",
)

add_subclass_annotation_inplace(
    A2,
    annotation_mapping_file,
    dataset_key="GEN_A2",
)

add_subclass_annotation_inplace(
    A3,
    annotation_mapping_file,
    dataset_key="GEN_A3",
)

[GEN_A1] mapped 9,929 / 10,032 cells; unmapped 103 (1.0%)
[GEN_A2] mapped 10,045 / 10,177 cells; unmapped 132 (1.3%)
[GEN_A3_ctx] mapped 526,244 / 530,170 cells; unmapped 3,926 (0.7%)


AnnData object with n_obs × n_vars = 526244 × 35116
    obs: 'Batch', 'rep', 'set', 'Source', 'brain_region', 'sample', 'individualID', 'libraryID', 'final_filt', 'Channel', 'n_genes', 'n_counts', 'percent_mito', 'passed_qc', 'G1/S', 'G2/M', 'cycle_diff', 'cycling', 'predicted_phase', 'female_score', 'male_score', 'gender_score', 'predicted_gender', 'mito_genes', 'mito_ribo', 'ribo_genes', 'apoptosis', 'class', 'subclass', 'subtype', 'doublet_score', 'pred_dbl', 'demux_type', 'study_id', 'subclass_annotation_v1'
    var: 'gene_symbols', 'feature_types', 'gene_id', 'gene_name', 'gene_type', 'gene_chrom', 'gene_start', 'gene_end', 'n_cells', 'percent_cells', 'robust', 'highly_variable_features', 'ribo', 'mito', 'protein_coding', 'mitocarta', 'robust_protein_coding', 'robust_protein_coding_autosome', 'mean', 'bins'

In [13]:
for adata in [A1, A2, A3]:
    adata.obs["cell_type"] = adata.obs["subclass_annotation_v1"].astype(str)

adatas["GEN_A1"] = A1
adatas["GEN_A2"] = A2
adatas["GEN_A3_ctx"] = A3

for key, adata in adatas.items():
    print(key, adata)

GEN_A1 AnnData object with n_obs × n_vars = 9929 × 35766
    obs: 'Batch', 'prep', 'rep', 'set', 'final_filt', 'individualID', 'Source', 'libraryID', 'Channel', 'n_genes', 'n_counts', 'percent_mito', 'passed_qc', 'G1/S', 'G2/M', 'cycle_diff', 'cycling', 'predicted_phase', 'female_score', 'male_score', 'gender_score', 'predicted_gender', 'mito_genes', 'mito_ribo', 'ribo_genes', 'apoptosis', 'class', 'subclass', 'subtype', 'doublet_score', 'pred_dbl', 'demux_type', 'study_id', 'subclass_annotation_v1', 'cell_type'
    var: 'gene_symbols', 'feature_types', 'gene_id', 'gene_name', 'gene_type', 'gene_chrom', 'gene_start', 'gene_end', 'n_cells', 'percent_cells', 'robust', 'highly_variable_features', 'ribo', 'mito', 'protein_coding', 'mitocarta', 'robust_protein_coding', 'robust_protein_coding_autosome', 'mean', 'bins'
GEN_A2 AnnData object with n_obs × n_vars = 10045 × 27203
    obs: 'n_genes', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_

In [14]:
def is_r_results_summary(path: Path) -> bool:
    return (
        path.name == "summary_stats_from_R.csv"
        and path.parent.name.startswith("R_results_")
    )


def extract_r_results_datetime(r_results_dir: str):
    """
    Extract timestamps like:
    R_results_2026-04-01_21-10
    """
    m = re.search(r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2})$", r_results_dir)

    if not m:
        return pd.NaT

    return pd.to_datetime(
        m.group(1),
        format="%Y-%m-%d_%H-%M",
        errors="coerce",
    )


def extract_target_pass(h5ad_path: str):
    """
    Extract pass number from filenames containing e.g. pass3.
    """
    if pd.isna(h5ad_path):
        return pd.NA

    m = re.search(r"pass(\d+)", Path(str(h5ad_path)).name)

    if not m:
        return pd.NA

    return int(m.group(1))


def get_target_h5ad(comparison_dir: Path):
    h5ad_inputs_file = comparison_dir / "h5ad_inputs.csv"

    if not h5ad_inputs_file.exists():
        return pd.NA, pd.NA

    h5ad_inputs = pd.read_csv(h5ad_inputs_file)
    target_rows = h5ad_inputs.loc[h5ad_inputs["h5ad_var"] == "target_h5ad"]

    if target_rows.empty:
        return pd.NA, pd.NA

    target_h5ad_path = target_rows.iloc[0]["h5ad_path"]
    target_pass = extract_target_pass(target_h5ad_path)

    return target_h5ad_path, target_pass

In [15]:
records = []

for search_dir in SEARCH_DIRS:
    for summary_csv in search_dir.rglob("summary_stats_from_R.csv"):
        if not is_r_results_summary(summary_csv):
            continue

        r_results_dir = summary_csv.parent
        comparison_dir = r_results_dir.parent

        subclass_map_path = r_results_dir / "assigned_subclasses_for_passing_subtypes.csv"
        target_h5ad_path, target_pass = get_target_h5ad(comparison_dir)

        comparison_prefix = re.sub(r"_vrs.*$", "", comparison_dir.name)

        records.append(
            {
                "comparison_prefix": comparison_prefix,
                "comparison_dir": comparison_dir.name,
                "comparison_dir_path": str(comparison_dir),
                "r_results_dir": r_results_dir.name,
                "r_results_dir_path": str(r_results_dir),
                "r_results_datetime": extract_r_results_datetime(r_results_dir.name),
                "target_h5ad_path": target_h5ad_path,
                "target_pass": target_pass,
                "subclass_map_path": (
                    str(subclass_map_path) if subclass_map_path.exists() else pd.NA
                ),
            }
        )

discovered = pd.DataFrame(records).drop_duplicates()

# Keep pass3 target_h5ad runs, matching the old script.
discovered = discovered.loc[discovered["target_pass"] == 3].copy()

# Keep only the newest R_results_* folder per comparison_dir.
newest = (
    discovered
    .sort_values(["comparison_dir", "r_results_datetime"])
    .groupby("comparison_dir", as_index=False)
    .tail(1)
    .reset_index(drop=True)
)

newest = newest[
    [
        "comparison_prefix",
        "comparison_dir",
        "r_results_dir",
        "target_h5ad_path",
        "subclass_map_path",
    ]
].sort_values("comparison_dir")

# Remove any datasets already loaded directly or equivalent to the old internal_dataset.
newest = newest.loc[
    ~newest["comparison_prefix"].astype(str).isin(EXCLUDE_DISCOVERED_STUDY_IDS)
].copy()

# Require real paths.
newest = newest.dropna(subset=["target_h5ad_path", "subclass_map_path"]).copy()

print(newest.to_string(index=False))

newest.to_csv(
    os.path.join(result_folder, "discovered_target_h5ad_and_subclass_map.csv"),
    index=False,
)

comparison_prefix                                     comparison_dir              r_results_dir                                                     target_h5ad_path                                                                                                                                                             subclass_map_path
          GEN_A13    GEN_A13_vrs_GEN_A1-3.cortex.subclass.1776491780 R_results_2026-04-20_20-50  /sbgenomics/project-files/GEN_A13/260212_GEN_A13/GEN_A13_pass3.h5ad    /sbgenomics/project-files/python_Metaneighbor_runs/GEN_A13_vrs_GEN_A1-3.cortex.subclass.1776491780/R_results_2026-04-20_20-50/assigned_subclasses_for_passing_subtypes.csv
          GEN_A16    GEN_A16_vrs_GEN_A1-3.cortex.subclass.1776500935 R_results_2026-04-20_20-47                 /sbgenomics/project-files/GEN_A16/GEN_A16_pass3.h5ad    /sbgenomics/project-files/python_Metaneighbor_runs/GEN_A16_vrs_GEN_A1-3.cortex.subclass.1776500935/R_results_2026-04-20_20-47/assigned_subclasses_for_pass

In [16]:
def load_target_h5ad_with_subclasses(
    target_h5ad_path,
    comparison_prefix,
    subclass_map_path,
    max_cells=None,
    random_seed=42,
    subtype_col="subtype",
    subclass_map_subtype_col="subtype",
    subclass_map_hit_col="best_subclass_hit",
    output_subclass_col="subclass",
    use_raw=True,
    set_gene_id_index=True,
):
    """
    Load one discovered target_h5ad and map its subtype labels to subclass labels
    using assigned_subclasses_for_passing_subtypes.csv.
    """
    target_h5ad_path = Path(target_h5ad_path)
    subclass_map_path = Path(subclass_map_path)

    if not target_h5ad_path.exists():
        raise FileNotFoundError(f"target_h5ad not found: {target_h5ad_path}")

    if not subclass_map_path.exists():
        raise FileNotFoundError(f"subclass map not found: {subclass_map_path}")

    backed = sc.read_h5ad(target_h5ad_path, backed="r")
    n_cells = backed.n_obs

    if max_cells is None or max_cells >= n_cells:
        cell_indices = np.arange(n_cells)
    else:
        rng = np.random.default_rng(random_seed)
        cell_indices = np.sort(
            rng.choice(n_cells, size=max_cells, replace=False)
        )

    if use_raw and backed.raw is not None:
        X = backed.raw.X[cell_indices, :]
        obs = backed.obs.iloc[cell_indices].copy()
        var = backed.var.copy()
        adata = sc.AnnData(X=X, obs=obs, var=var)
    else:
        adata = backed[cell_indices, :].to_memory()

    backed.file.close()

    adata = adata.to_memory()

    if set_gene_id_index and "gene_id" in adata.var.columns:
        adata.var = adata.var.set_index("gene_id", drop=False)

    adata.obs["study_id"] = comparison_prefix

    subclass_map = pd.read_csv(subclass_map_path)

    required_map_cols = {subclass_map_subtype_col, subclass_map_hit_col}
    missing_map_cols = required_map_cols - set(subclass_map.columns)

    if missing_map_cols:
        raise ValueError(
            f"Missing columns in subclass map: {sorted(missing_map_cols)}"
        )

    if subtype_col not in adata.obs.columns:
        raise ValueError(
            f"{subtype_col!r} not found in adata.obs for {comparison_prefix}. "
            f"Available obs columns include: {list(adata.obs.columns[:20])}"
        )

    subtype_to_subclass = (
        subclass_map
        .drop_duplicates(subset=[subclass_map_subtype_col])
        .assign(**{
            subclass_map_subtype_col: lambda x: x[subclass_map_subtype_col].astype(str)
        })
        .set_index(subclass_map_subtype_col)[subclass_map_hit_col]
    )

    adata.obs[output_subclass_col] = (
        adata.obs[subtype_col]
        .astype(str)
        .map(subtype_to_subclass)
    )

    matched_cells = adata.obs[output_subclass_col].notna()
    n_missing = int((~matched_cells).sum())

    if n_missing > 0:
        print(
            f"[{comparison_prefix}] dropping {n_missing:,} / {adata.n_obs:,} cells "
            f"that did not map from obs[{subtype_col!r}] to subclass."
        )

    adata = adata[matched_cells].copy()
    adata.obs["cell_type"] = adata.obs[output_subclass_col].astype(str)

    return adata

In [17]:
for row in newest.itertuples(index=False):
    print(f"Loading {row.comparison_prefix}")

    if row.comparison_prefix in adatas:
        print(f"Skipping {row.comparison_prefix}; already loaded.")
        continue

    adatas[row.comparison_prefix] = load_target_h5ad_with_subclasses(
        target_h5ad_path=row.target_h5ad_path,
        comparison_prefix=row.comparison_prefix,
        subclass_map_path=row.subclass_map_path,
        max_cells=MAX_CELLS_PER_DISCOVERED_TARGET,
        random_seed=RANDOM_SEED,
    )

for key, adata in adatas.items():
    if "cell_type" not in adata.obs.columns:
        raise ValueError(f"{key}: obs['cell_type'] not found")

    print(key, adata.n_obs, adata.n_vars, adata.obs["cell_type"].nunique())

Loading GEN_A13
[GEN_A13] dropping 113 / 10,000 cells that did not map from obs['subtype'] to subclass.
Loading GEN_A16
[GEN_A16] dropping 57 / 10,000 cells that did not map from obs['subtype'] to subclass.
Loading GEN_A4_ITG
[GEN_A4_ITG] dropping 354 / 10,000 cells that did not map from obs['subtype'] to subclass.
Loading GEN_A4_MEC
[GEN_A4_MEC] dropping 97 / 10,000 cells that did not map from obs['subtype'] to subclass.
Loading GEN_A4_MTG
[GEN_A4_MTG] dropping 130 / 10,000 cells that did not map from obs['subtype'] to subclass.
Loading GEN_A4_PFC
[GEN_A4_PFC] dropping 166 / 10,000 cells that did not map from obs['subtype'] to subclass.
Loading GEN_A4_PVC
[GEN_A4_PVC] dropping 697 / 10,000 cells that did not map from obs['subtype'] to subclass.
Loading GEN_A4_STG


KeyboardInterrupt: 

In [ ]:
direct_inputs = pd.DataFrame(
    {
        "h5ad_var": ["A1_h5ad", "A2_h5ad", "A3_h5ad"],
        "dataset_name": ["GEN_A1", "GEN_A2", "GEN_A3_ctx"],
        "sample_fraction": [
            GEN_A1_SAMPLE_FRACTION,
            GEN_A2_SAMPLE_FRACTION,
            GEN_A3_SAMPLE_FRACTION,
        ],
        "max_cells": [pd.NA, pd.NA, pd.NA],
        "h5ad_path": [A1_h5ad, A2_h5ad, A3_h5ad],
        "subclass_source": [
            annotation_mapping_path,
            annotation_mapping_path,
            annotation_mapping_path,
        ],
    }
)

discovered_inputs = newest.rename(
    columns={
        "comparison_prefix": "dataset_name",
        "target_h5ad_path": "h5ad_path",
        "subclass_map_path": "subclass_source",
    }
).copy()

discovered_inputs["h5ad_var"] = "discovered_target_h5ad"
discovered_inputs["sample_fraction"] = pd.NA
discovered_inputs["max_cells"] = MAX_CELLS_PER_DISCOVERED_TARGET

discovered_inputs = discovered_inputs[
    [
        "h5ad_var",
        "dataset_name",
        "sample_fraction",
        "max_cells",
        "h5ad_path",
        "subclass_source",
        "comparison_dir",
        "r_results_dir",
    ]
]

h5ad_inputs = pd.concat(
    [direct_inputs, discovered_inputs],
    ignore_index=True,
    sort=False,
)

h5ad_inputs.to_csv(
    os.path.join(result_folder, "h5ad_inputs.csv"),
    index=False,
)

print(h5ad_inputs.to_string(index=False))

In [ ]:
print("Datasets to merge:")
for key, adata in adatas.items():
    print(f"{key}: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

merged = ad.concat(adatas, join="inner")

print(merged)

In [ ]:
# Clear individual AnnData objects after merging
del adatas

# Also clear direct references to A1/A2/A3 if they exist
for obj_name in ["A1", "A2", "A3"]:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()

In [ ]:
print("Before running HVGs")

merged.obs.index = merged.obs.index.to_numpy(dtype="str")

pymn.variableGenes(
    merged,
    study_col="study_id",
)

print(f"Number of HVGs: {int(merged.var['highly_variable'].sum()):,}")

In [ ]:
hvg_mask = merged.var["highly_variable"].to_numpy(dtype=bool)

merged_hvg = merged[:, hvg_mask].copy()

del merged
gc.collect()

merged = merged_hvg

del merged_hvg
gc.collect()

print(merged)

In [ ]:
# Use numpy types instead of pandas objects where possible.
merged.var["highly_variable"] = merged.var["highly_variable"].to_numpy(dtype="bool")
merged.var.index = merged.var.index.to_numpy(dtype="str")

merged.obs_names_make_unique()

merged.obs["cell_type"] = merged.obs["cell_type"].to_numpy(dtype="str")
merged.obs["study_id"] = merged.obs["study_id"].to_numpy(dtype="str")
merged.obs.index = merged.obs.index.to_numpy(dtype="str")

In [ ]:
print("Before running MetaNeighbor all-vs-all")

pymn.MetaNeighborUS(
    merged,
    study_col="study_id",
    ct_col="cell_type",
    fast_version=True,
    symmetric_output=True,
)

print("After running MetaNeighbor all-vs-all")

aurocs = merged.uns["MetaNeighborUS"]

aurocs.to_csv(
    os.path.join(result_folder, "aurocs_full.csv.gz"),
    compression="gzip",
)

In [ ]:
print("Before running MetaNeighbor one-vs-best")

pymn.MetaNeighborUS(
    merged,
    study_col="study_id",
    ct_col="cell_type",
    one_vs_best=True,
    fast_version=True,
    symmetric_output=True,
)

print("After running MetaNeighbor one-vs-best")

aurocs_1v1 = merged.uns["MetaNeighborUS_1v1"]

aurocs_1v1.to_csv(
    os.path.join(result_folder, "aurocs_1v1.csv.gz"),
    compression="gzip",
)

In [ ]:
cell_counts = merged.obs.groupby("study_id").size()

cell_counts.to_csv(
    os.path.join(result_folder, "cell_study_counts.csv")
)

cell_type_counts = (
    merged.obs[["study_id", "cell_type"]]
    .drop_duplicates()
    .groupby("study_id")
    .size()
)

cell_type_counts.to_csv(
    os.path.join(result_folder, "cell_type_per_study_counts.csv")
)

print(cell_counts)
print(cell_type_counts)

In [ ]:
for set_threshold in [0.8, 0.9, 0.95, 0.99, 0.999]:
    print(set_threshold)

    pymn.topHits(
        merged,
        threshold=set_threshold,
    )

    tophit_table = merged.uns["MetaNeighborUS_topHits"]

    tophit_table.to_csv(
        os.path.join(result_folder, f"top_hits.{set_threshold}.csv")
    )

In [ ]:
merged.obs.to_csv(
    os.path.join(result_folder, "merged.obs.csv.gz"),
    compression="gzip",
)

merged.var.to_csv(
    os.path.join(result_folder, "merged.var.csv.gz"),
    compression="gzip",
)

In [ ]:
mem_usage = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss

peak_memory_gb = round(mem_usage / 1024 / 1024, 2)

print(
    "Peak memory use in Gb  "
    + str(peak_memory_gb)
    + " PID  "
    + str(os.getpid())
)

os.mkdir(
    os.path.join(
        result_folder,
        "Peak memory use in Gb " + str(peak_memory_gb),
    )
)

In [ ]:
end_time = time.time()

time_taken = (
    str(datetime.timedelta(seconds=end_time - start_time))
    .replace(":", "_")
    .split(".")[0]
)

print("Time taken h_m_s " + time_taken)

os.mkdir(
    os.path.join(
        result_folder,
        "Time taken h_m_s " + time_taken,
    )
)

print("Done!")
print(result_folder)